In [2]:
import cv2
import mediapipe as mp
import time

In [3]:
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

In [4]:
# For webcam input:
video_path = 'D:\Ramana\Carrer\Projects\CV Gun Game\InputVid1.mp4'
cap = cv2.VideoCapture(0)

In [5]:
# Variables to track hand position and state
prev_x = None
hand_open = False

In [6]:
def is_hand_open(landmarks):
    # Check if hand is open by comparing finger tip positions with their base
    # Using thumb, index, and middle finger as indicators
    thumb_tip = landmarks[4].y
    thumb_base = landmarks[2].y
    index_tip = landmarks[8].y
    index_base = landmarks[5].y
    middle_tip = landmarks[12].y
    middle_base = landmarks[9].y
    
    # Hand is considered open if fingers are extended (tips higher than bases)
    return (thumb_tip < thumb_base and 
            index_tip < index_base and 
            middle_tip < middle_base)

In [7]:
while True:
    success, frame = cap.read()
    if not success:
        break
        
    # Flip the frame horizontally for natural movement
    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process the frame
    results = hands.process(frame_rgb)
    
    if results.multi_hand_landmarks:
        for hand_landmarks, handedness in zip(results.multi_hand_landmarks, 
                                            results.multi_handedness):
            # Check if it's the left hand
            if handedness.classification[0].label == "Left":
                # Draw hand landmarks
                mp_draw.draw_landmarks(frame, hand_landmarks, 
                                     mp_hands.HAND_CONNECTIONS)
                
                # Check if hand is open
                hand_open = is_hand_open(hand_landmarks.landmark)
                
                if hand_open:
                    # Get wrist x-coordinate (landmark 0)
                    current_x = hand_landmarks.landmark[0].x * frame.shape[1]
                    
                    if prev_x is not None:
                        # Compare current position with previous position
                        if current_x < prev_x - 20:  # Moved left (threshold of 20 pixels)
                            direction = "backwards time"
                            #print("backwards time")                            
                        elif current_x > prev_x + 20:  # Moved right
                            #print("forward time")
                            direction = "forward time"
                        else:
                            direction = "no time movement"
                            #print("no time movement")
                    
                    prev_x = current_x
                else:
                    direction = "no time movement"
                    #print("no time movement")
                    prev_x = None
                    
    else:        
        direction = "no time movement"
        #print("no time movement")
        prev_x = None

 # Display direction on the frame
    if direction:
         # Put a background rectangle for better text visibility
        cv2.rectangle(frame, (10, 10), (200, 60), (0, 0, 0), -1)
        # Display the direction text
        cv2.putText(frame, f"Direction: {direction}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        # Display "No direction detected" when no hand is found
        cv2.rectangle(frame, (10, 10), (300, 60), (0, 0, 0), -1)
        cv2.putText(frame, "No direction detected", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Display the frame
    cv2.imshow("Hand Tracking", frame)
    
    # Break loop on 'q' press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


: 

In [32]:
# Release resources
cap.release()
cv2.destroyAllWindows()
hands.close()

In [1]:
import cv2
import mediapipe as mp
import time

In [2]:

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

In [3]:

# For webcam input:
cap = cv2.VideoCapture(0)  # Note: Your video_path variable wasn't used, so I kept it as webcam

In [4]:

# Variables to track hand position and state
prev_x = None
last_direction = "no time movement"
movement_threshold = 5  # Reduced threshold for slower movements (adjust as needed)
velocity_window = []  # To track recent positions for smoother detection

In [5]:
def is_hand_open(landmarks):
    # Check if hand is open by comparing finger tip positions with their base
    thumb_tip = landmarks[4].y
    thumb_base = landmarks[2].y
    index_tip = landmarks[8].y
    index_base = landmarks[5].y
    middle_tip = landmarks[12].y
    middle_base = landmarks[9].y
    return (thumb_tip < thumb_base and 
            index_tip < index_base and 
            middle_tip < middle_base)

In [6]:
while True:
    success, frame = cap.read()
    if not success:
        break
        
    # Flip the frame horizontally for natural movement
    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process the frame
    results = hands.process(frame_rgb)
    
    direction = last_direction  # Default to last direction
    
    if results.multi_hand_landmarks:
        for hand_landmarks, handedness in zip(results.multi_hand_landmarks, 
                                            results.multi_handedness):
            # Check if it's the left hand
            if handedness.classification[0].label == "Left":
                # Draw hand landmarks
                mp_draw.draw_landmarks(frame, hand_landmarks, 
                                     mp_hands.HAND_CONNECTIONS)
                
                # Check if hand is open
                hand_open = is_hand_open(hand_landmarks.landmark)
                
                if hand_open:
                    # Get wrist x-coordinate (landmark 0)
                    current_x = hand_landmarks.landmark[0].x * frame.shape[1]
                    
                    if prev_x is not None:
                        # Calculate movement
                        movement = current_x - prev_x
                        
                        # Add to velocity window (keep last 5 frames)
                        velocity_window.append(movement)
                        if len(velocity_window) > 5:
                            velocity_window.pop(0)
                        
                        # Average movement over window for smoother detection
                        avg_movement = sum(velocity_window) / len(velocity_window)
                        
                        # Detect direction with lower threshold
                        if avg_movement < -movement_threshold:  # Moved left
                            direction = "backwards time"
                        elif avg_movement > movement_threshold:  # Moved right
                            direction = "forward time"
                        elif abs(avg_movement) < movement_threshold / 2:  # Nearly still
                            direction = "no time movement"
                        # If movement is small but non-zero, keep last direction
                        
                    prev_x = current_x
                else:
                    direction = "no time movement"
                    prev_x = None
                    velocity_window.clear()
                    
    else:
        direction = "no time movement"
        prev_x = None
        velocity_window.clear()

    # Update last direction only if we have a new definitive direction
    if direction in ["backwards time", "forward time", "no time movement"]:
        last_direction = direction

    # Display direction on the frame
    if direction:
        # Put a background rectangle for better text visibility
        cv2.rectangle(frame, (10, 10), (300, 60), (0, 0, 0), -1)
        # Display the direction text
        cv2.putText(frame, f"Direction: {direction}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        # Display "No direction detected" when no hand is found
        cv2.rectangle(frame, (10, 10), (300, 60), (0, 0, 0), -1)
        cv2.putText(frame, "No direction detected", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Display the frame
    cv2.imshow("Hand Tracking", frame)
    
    # Break loop on 'q' press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [7]:
# Release resources
cap.release()
cv2.destroyAllWindows()
hands.close()